# Data Scraping, Intro - Lane B (backup: the agent's output, captured)

**SISMID 2026 - Day 1, 9:45 session.**

> **What this notebook is.** Every cell below is a captured example of what a coding
> agent (Codex, Claude Code, or Antigravity CLI) produces when you give it the
> prompts in the **Lane A** notebook. Lane A is the real experience: you prompt, the
> agent writes this. Lane B is here as a **backup** (if your agent is not set up yet)
> and as a **reference** for what good output looks like. The prompt that produced each
> cell is named in a comment at the top of the cell.

Target: **Google Dengue Trends for Mexico**. Two pulls:
1. a small **live** pull of a handful of terms over the **past 5 years**, so you see
   current dengue searches and learn the flow, and
2. the **historical** 2004-2011 window that feeds the 11:00 exercise (cached, so it
   always works even when Google rate-limits the room).


## Step 0: the helper (from the Step 0 prompt)


In [ ]:
# --- Produced by the agent from the Plan A / Step 0 prompt ---
from pytrends.request import TrendReq   # pip install pytrends (in requirements.txt)
import pandas as pd, matplotlib.pyplot as plt, os, time, random

GEO = 'MX'                       # Mexico, to match the 11:00 Dengue exercise

# --- the LIVE flow: a handful of terms, past 5 years up to today ---
LIVE_TERMS = ['dengue', 'sintomas de dengue', 'mosquito']   # a handful, max 5
RECENT_TF  = 'today 5-y'         # past 5 years; the last point is the current week

# --- the historical window the 11:00 exercise needs (cached, always works) ---
HIST_TERMS = ['dengue', 'sintomas de dengue', 'mosquito', 'dengue sintomas']
HIST_TF    = '2004-01-01 2011-12-31'

CACHE_PATHS = [
    '../data/google_trends_dengue_mx_cached.csv',
    'data/google_trends_dengue_mx_cached.csv',
    './google_trends_dengue_mx_cached.csv',
]

def _norm(c):
    return c.strip().replace(' ', '_')

def gt_fetch(kw_list, timeframe, geo=GEO, tries=4):
    """Google Trends interest-over-time for a handful of terms (max 5).
    Returns a tidy DataFrame (date + one column per term), or None if Google
    keeps refusing. A first 429 is normal even from a Codespace (Azure IP);
    we wait and retry, which is exactly what got our smoke test through. The
    small random pause staggers the room so we do not all hit Google at once."""
    time.sleep(random.uniform(0, 3))   # stagger: do NOT count '3-2-1-everyone run'
    for attempt in range(tries):
        try:
            pt = TrendReq(hl='en-US', tz=360, retries=2, backoff_factor=0.5)
            pt.build_payload(kw_list, timeframe=timeframe, geo=geo)
            df = pt.interest_over_time()
            if df.empty:
                raise RuntimeError('empty frame (rate-limited)')
            df = df.drop(columns=[c for c in ['isPartial'] if c in df.columns]).reset_index()
            return df.rename(columns={c: _norm(c) for c in df.columns})
        except Exception as e:
            rate_limited = ('429' in str(e) or 'empty frame' in str(e)
                            or 'toomany' in type(e).__name__.lower())
            if rate_limited and attempt < tries - 1:
                wait = 12 * (attempt + 1)
                print(f'Rate-limited (attempt {attempt+1}/{tries}); waiting {wait}s and retrying...')
                time.sleep(wait)
                continue
            print(f'Live Google Trends pull failed ({type(e).__name__}): {e}')
            return None
    return None

def load_hist_cache():
    for p in CACHE_PATHS:
        if os.path.exists(p):
            print(f'Using cached historical snapshot: {p}')
            return pd.read_csv(p, parse_dates=['date'])
    raise FileNotFoundError('Historical cache not found; check the data/ folder.')


## Step 1: see it *live* (a handful of terms, past 5 years)

The whole live flow in one call: three dengue-related terms for Mexico over the last
five years. The last date lands on the **current week**. A first 429 is normal;
`gt_fetch` waits and retries automatically. `None` means it gave up after several
tries: rerun in a minute, or continue to the cached section below.


In [ ]:
# --- Produced by the agent from the Plan A / Step 1 prompt ---
live = gt_fetch(LIVE_TERMS, timeframe=RECENT_TF)
if live is not None:
    print('rows:', len(live), '| last data point:', live['date'].max().date())
    cols = [c for c in live.columns if c != 'date']
    plt.figure(figsize=(10,4))
    for c in cols:
        plt.plot(live['date'], live[c], label=c)
    plt.legend(); plt.title('Google Trends (live): dengue terms in Mexico, past 5 years')
    plt.ylabel('search interest (0-100)'); plt.xlabel('date'); plt.tight_layout(); plt.show()
    display(live.tail())


## Step 2: the instability note (from the Step 2 prompt)

Pull the same term twice. Google Trends is built from a **sample** of searches, so
pulls can differ: usually only slightly on the public 0-100 endpoint, but substantially
on the raw Google Health Trends API. Either way, this is why the afternoon session is
about statistical preprocessing.


In [ ]:
# --- Produced by the agent from the Plan A / Step 2 prompt ---
a = gt_fetch(['dengue'], timeframe=RECENT_TF)
b = gt_fetch(['dengue'], timeframe=RECENT_TF)
if a is not None and b is not None:
    m = a.merge(b, on='date', suffixes=('_1','_2'))
    diff = (m['dengue_1'] - m['dengue_2']).abs()
    print('identical pulls? ', bool((diff == 0).all()))
    print('mean abs difference:', round(diff.mean(), 2))
else:
    print('Could not compare live (rate-limited). The point stands: repeat pulls can differ.')


## Step 3: the historical window for 11:00 (from the Step 3 prompt)

The four dengue-related terms over 2004-2011, matched to the reported-case ground
truth you will use next session. Try live, then fall back to the cache, so this cell
**always** ends with usable data.


In [ ]:
# --- Produced by the agent from the Plan A / Step 3 prompt ---
hist = gt_fetch(HIST_TERMS, timeframe=HIST_TF)
if hist is None:
    hist = load_hist_cache()
hist.head()


In [ ]:
cols = [c for c in hist.columns if c != 'date']
plt.figure(figsize=(10,4))
for c in cols:
    plt.plot(hist['date'], hist[c], label=c)
plt.legend(); plt.title('Dengue-related searches in Mexico, 2004-2011')
plt.ylabel('search interest'); plt.xlabel('date'); plt.tight_layout(); plt.show()


## Step 4: sanity-check and save (from the Step 4 prompt)

Cheap checks earn a stream its place in a model, then save the tidy CSV that feeds 11:00.


In [ ]:
# --- Produced by the agent from the Plan A / Step 4 prompt ---
print('date range :', hist['date'].min().date(), 'to', hist['date'].max().date())
print('n rows     :', len(hist))
print('missing per column:'); print(hist.isna().sum())
print('which terms actually move (nonzero variance)?'); print(hist[cols].std().round(2))
hist.to_csv('dengue_mx_search.csv', index=False)
print('saved dengue_mx_search.csv with', len(hist), 'rows and columns', list(hist.columns))


## Reflection

- Everything above is what the agent writes from a few plain-English prompts. That is
  the point of Lane A; this notebook just captures it so nobody is blocked.
- You ran the **live flow** (a handful of terms, past 5 years) and saw *current*
  searches, then pulled the **historical** window the model needs.
- Live pulls succeed when **small and paced**; they 429 when everyone bursts at once,
  which is why the cache is the guaranteed fallback.

**Stretch:** change `GEO` or swap `LIVE_TERMS` for a disease you care about (keep it to
a handful), and rerun.
